# A.7 多輪對話和答案怎麼分配長度？


共用100個位置的完整歷史生成，已放歷史20、目前問題與格式15、公告40，輸入用75，還能預留25個新token。若歷史增加30，輸入本身就達105：即使完全不回答，也超出5個位置。因此必須先縮短輸入；若仍想保留25個回答位置，就需從輸入刪去30個位置，不能只縮回答上限來解決。

此例假定decoder-only推論保留完整歷史，輸入P與新輸出預留G共用總上限C，按 `P+G≤C` 算。對話角色和分隔格式也佔位置；token由實際tokenizer切分，不能直接用中文字數取代。先把系統、歷史、問題與公告序列化成最終輸入，再數ID。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
budget = {"history": 20, "question_and_format": 15, "retrieval": 40, "answer": 25}
limit = 100
used = sum(budget.values())
print("輸入", used - budget["answer"], "答案預留", budget["answer"])
print("總預算", used, "放得下", used <= limit)
budget["history"] += 30
used = sum(budget.values())
print("歷史變長後", used, "超出", max(0, used - limit))

前兩行是輸入75、回答預留25、總100；歷史多30後總130、超30。這些是人工token數的算術，不是程式已經裁掉歷史。回答預留也只是上限，EOS可提早結束。

歷史可移除不相關舊回合，但仍在使用的店名與條件要留下。公告可選短片段，卻不能切掉地址、日期或引用；少留回答空間則可能在答案未完時截停。摘要也會漏資訊，因此裁切後重新判斷資料是否足以答題。

把公告40改成10，總數回100，但還要能指出拿掉的30格為何不影響證據。算術上放得下與證據足夠分開核對；[7.19](https://birdhackor.github.io/tiny-perceptron-vlm/7.19.html)補充角色和生成的序列位置。即使預算合格，遠處線索也未必被穩定使用，下一節只移動同一線索來看這件事。

<details>
<summary>補充：舊短RAG輸入與停止紀錄</summary>

本輪84份RAG生成的完整序列化輸入各用58至94個token，另預留最多16個新增token，均放得進160位置；沒有靠切掉公告完成評分。全部84份都有EOS，沒有非法控制token，內容仍會失敗。報告保存`input_ids`、`generated_ids`、生成上限與EOS；先檢查原始ID中的控制標記，再比對回答文字，不讓解碼隱藏角色標記而誤計成功。本輪沒有長歷史或摘要實驗，這些短題的停止率不能替代長上下文品質評測。

</details>
